# 군사기지 주변 지역의 주택 가격 예측하기

이 문제는 목표는 가상의 군사기지 주변 지역 정보를 활용해 그 지역의 주택 가격의 중간값을 예측하는 Regression 모델을 만드는 것입니다.  


## 데이터 셋 설명

주택 가격을 예측하는 학습을 하기 위한 훈련용 데이터와 나중에 만들어진 모델의 성능을 측정하기 위한 테스트 데이터가 별도의 파일로  제공이 됩니다.

 - 훈련용 데이터: train.csv
 - 테스트용 데이터: test.csv

훈련용 데이터와 테스트 데이터의 차이점은, 테스트 데이터의 경우 주택 가격의 중간값을 나타내는 median_house_value란 레이블 필드가 없고 일련번호 필드인 id가 존재한다는 점입니다.

### 훈련용 데이터

다음은 데이터셋(train.csv)에 들어있는 피쳐들과 그에 대한 설명들입니다. 이 정보는 주택별 정보가 아닌 지역별 정보입니다. 이를 바탕으로 지역별 주택 중간값을 예측합니다.

| 필드 | 타입 | 설명 |
| --- | --- | --- |
| housing_median_age | 숫자 | 해당 지역 주택 나이의 중간값  |
| total_rooms | 숫자 | 해당 지역 주택의 총 방의 수  |
| total_bedrooms | 숫자 | 해당 지역 주택의 총 침실의 수  |
| population | 숫자 | 해당 지역 인구 |
| households | 숫자 | 해당 지역 가구수 |
| median_income | 숫자 | 해당 지역 가구의 중간 소득 |
| ocean_proximity | 문자열 | 해당 지역으로부터 바닷가까지의 거리를 나타내는 문자열로 다음과 같은 값이 가능.<br> <left><li> "<1H OCEAN": 바다에서 1시간 이내의 거리에 있는 지역<br> <li> "INLAND": 내륙 지역으로 바다에서 멀리 떨어져 있는 지역<br><li>  "NEAR BAY": 만 근처에 있는 지역으로, 바다보다는 좀 더 작은 규모의 해안 지역이나 만과 가까운 위치<br> <li> "NEAR OCEAN": 바다와 가까운 지역으로, 해안 근처에 있는 주택</left>  |
| median_house_value | 숫자 | 예측 대상이 되는 주택 가격의 중간값 |

### 테스트용 데이터

아래는 테스트 데이터인 test.csv 파일의 예시입니다. train.csv 파일과 비교했을 때, 레이블 필드인 median\_house\_value가 없고 일련번호 필드인 id가 존재합니다.

| id |housing_median_age |total_rooms |total_bedrooms |population |households |median_income |ocean_proximity |
|  --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 9.0 | 5206.0 | 992.0 | 4660.0 | 978.0 | 2.885 | <1H OCEAN |
| 1 | 28.0 | 2608.0 | 576.0 | 1719.0 | 554.0 | 2.1186 | INLAND |
| 2 | 33.0 | 2731.0 | 456.0 | 1263.0 | 445.0 | 4.5568 | NEAR OCEAN |
| 3 | 36.0 | 1406.0 | 224.0 | 598.0 | 237.0 | 5.8964 | NEAR OCEAN |
| 4 | 18.0 | 1525.0 | 485.0 | 1705.0 | 448.0 | 3.7198 | <1H OCEAN |

### 초기코드 선택(Python)

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 초기코드 - python
import pandas as pd

# 데이터 로드
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test.csv')

train.head()

,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,median_house_value
0,43.0,1461.0,284.0,800.0,258.0,3.2279,<1H OCEAN,182400.0
1,33.0,2613.0,562.0,3150.0,543.0,4.3899,<1H OCEAN,180700.0
2,11.0,5363.0,1160.0,2783.0,1034.0,3.8583,<1H OCEAN,170700.0
3,43.0,2700.0,510.0,1146.0,506.0,5.1333,<1H OCEAN,345900.0
4,36.0,1407.0,385.0,1763.0,350.0,2.6364,<1H OCEAN,150000.0


### 제출 데이터

submission.csv 파일은 CSV 포맷으로 두 개의 필드로 구성됩니다 (id와 median_house_value). 아래 예가 있습니다.

|id|median_house_value|
|-|-|
|0|414800|
|1|294700|
|2|150000|
|3|78600|
|4|94400|

### 성능 평가

모델의 성능은 RMSE (Root Mean Square Error)으로 평가합니다. 70000 보다 작은 값이 나오면 괜찮다고 생각할 수 있습니다.

In [74]:
# 여기에 코드를 작성하세요.
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test.csv')

target = 'median_house_value'
num_cols = ['housing_median_age', 'total_rooms', 'total_bedrooms',
            'population', 'households', 'median_income']
cat_col = 'ocean_proximity'

# ---------- 1. 전처리 ----------
# ---------- 1. 전처리 (맨 앞: 클리핑/스케일링 이전) ----------
for df_ in (train, test):
    df_['rooms_per_household'] = df_['total_rooms'] / df_['households']
    df_['bedrooms_per_room'] = df_['total_bedrooms'] / df_['total_rooms']
    df_['population_per_household'] = df_['population'] / df_['households']

# 새 변수도 클리핑/정규화 대상에 포함
num_cols = ['housing_median_age', 'total_rooms', 'total_bedrooms',
            'population', 'households', 'median_income',
            'rooms_per_household', 'bedrooms_per_room', 'population_per_household']
# (2) 이상치 클리핑: 경계는 train에서만 계산해서 train/test에 동일하게 적용
lower = train[num_cols].quantile(0.01)   # 가이드: 0.01 ~ 0.05 사이에서 선택
upper = train[num_cols].quantile(0.99)   # 가이드: 0.95 ~ 0.99 사이에서 선택
train[num_cols] = train[num_cols].clip(lower, upper, axis=1)
test[num_cols] = test[num_cols].clip(lower, upper, axis=1)

# (3) 정규화: scaler는 train에만 fit (test 정보 누수 방지)
scaler = MinMaxScaler()                 # 표준 스케일링을 원하면 StandardScaler로 교체
train[num_cols] = scaler.fit_transform(train[num_cols])
test[num_cols] = scaler.transform(test[num_cols])

# (4) 범주형 인코딩: 원-핫 (질문에서 미정이라 임의로 선택, 바꾸고 싶으면 알려주세요)
train = pd.get_dummies(train, columns=[cat_col], dtype=int)
test = pd.get_dummies(test, columns=[cat_col], dtype=int)

# XGBoost는 컬럼명에 [, ], < 를 허용하지 않으므로 치환
for df_ in (train, test):
    df_.columns = (df_.columns
                   .str.replace('<', 'lt_', regex=False)
                   .str.replace('[', '(', regex=False)
                   .str.replace(']', ')', regex=False))

feature_cols = [c for c in train.columns if c != target]
X, y = train[feature_cols], train[target]
X_test = test[feature_cols]             # test의 id는 feature에서 제외 (별도 보관)

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.3, random_state=10)

# ---------- 2. 모델 ----------
def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

# (A) Linear Regression
lr = LinearRegression()              # 가이드: 기본값으로 충분, 필요 시 fit_intercept만
lr.fit(X_tr, y_tr)
print('LR RMSE:', rmse(y_val, lr.predict(X_val)))

# (B) XGBoost
xgb = XGBRegressor(
    n_estimators=300,                   # 가이드: 수백~천 단위, early stopping과 함께
    learning_rate=0.0432,                  # 가이드: 0.01 ~ 0.1
    max_depth=5,                      # 가이드: 3 ~ 8
    subsample=0.75,                      # 가이드: 0.6 ~ 1.0
    colsample_bytree=0.95,               # 가이드: 0.6 ~ 1.0
    random_state=10,
)
lgbm = LGBMRegressor(
    n_estimators=300, 
    learning_rate=0.0432, 
    num_leaves=33,   # 가이드: 수백~천 / 0.01~0.1 / 15~63
    min_child_samples=25,                                 # 가이드: 10~50 (작은 데이터는 크게)
    subsample=0.75, colsample_bytree=0.95, random_state=10, verbose=-1,
)

rf = RandomForestRegressor(
    n_estimators=300,                                      # 가이드: 200~500
    max_depth=11, min_samples_leaf=3,                   # 가이드: None 또는 10~20 / 1~5
    max_features=0.75, n_jobs=-1, random_state=10,         # 가이드: 0.5~1.0
)

# ---------- 개별 모델 검증 ----------
models = {'xgb': xgb, 'lgbm': lgbm, 'rf': rf}
val_preds = {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    val_preds[name] = m.predict(X_val)
    print(f'{name} RMSE:', rmse(y_val, val_preds[name]))

# ---------- 앙상블 검증 ----------
# (A) 단순 평균
avg_pred = np.mean(list(val_preds.values()), axis=0)
print('Ensemble (mean) RMSE:', rmse(y_val, avg_pred))

# (B) 가중 평균: 가중치 합은 1, 검증 RMSE가 낮은 모델에 크게
w = {'xgb': 0.3, 'lgbm': 0.2, 'rf': 0.5}
w_pred = sum(w[k] * val_preds[k] for k in models)
print('Ensemble (weighted) RMSE:', rmse(y_val, w_pred))

# ---------- 최종 제출: 전체 데이터로 재학습 후 앙상블 ----------
test_preds = {}
for name, m in models.items():
    m.fit(X, y)
    test_preds[name] = m.predict(X_test)

final_pred = np.mean(list(test_preds.values()), axis=0)   # 가중 평균이 더 좋으면 (B) 방식으로 교체

submission = pd.DataFrame({'id': test['id'], 'median_house_value': final_pred})
submission.to_csv('submission.csv', index=False)

LR RMSE: 64812.375456474925
xgb RMSE: 59585.32759170501
lgbm RMSE: 60324.266805576524
rf RMSE: 59284.000001751985
Ensemble (mean) RMSE: 58940.1799998367
Ensemble (weighted) RMSE: 58839.05048403916


### 결과 저장

채점을 위해, 위에서 구한 데이터를 현재 파일과 같은 디렉토리(.ipynb 파일이 있는 디렉토리)에 `submission.csv`이라는 이름으로 저장해주세요.

In [51]:
# 여기에 코드를 작성하세요.

In [52]:
# csv 파일 저장 예시 - python
#df.to_csv('submission.csv')

### 결과 테스트

해당 파트는 실제 예선전과 별개로 성능 확인을 위해 추가된 셀입니다.

In [75]:
from sklearn.metrics import mean_squared_error

# 1. CSV 파일 불러오기
df1 = pd.read_csv('./data/test_with_labels.csv')
df2 = pd.read_csv('submission.csv')

# 2. 마지막 컬럼 추출
col1 = df1.iloc[:, -1]
col2 = df2.iloc[:, -1]

# 3. RMSE 계산
rmse = np.sqrt(mean_squared_error(col1, col2))

# 4. 결과 출력
print("RMSE:", rmse)

RMSE: 57830.13733021038
